# ⚾ O.R.2 프로야구(1982 KBO 6개 구단) 10일 경기 일정 최적화 — 규민 개량본 & 원본 (Colab 연동)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gyumin225/OR2-project/blob/main/OR2_KBO_GA_Colab.ipynb)

이 노트북은 GitHub 저장소 [`Gyumin225/OR2-project`](https://github.com/Gyumin225/OR2-project)와 연동되어 있으며, **Google Colab에서 바로 실행**할 수 있도록 구성되었습니다.

### 📊 핵심 비교 요약 (시드 0–29, 공통 평가기 기준)
| 설정 | 코드 | 평균 C ± SD | 최선 C(x) | 예시(2.013)보다 좋은 시드 |
| --- | --- | --- | --- | --- |
| 동일 예산 17,460회 | 규민 원본 (`pc=0.9`) | 1.988 ± 0.112 | 1.824 (`seed=3`) | 20/30 |
| 동일 예산 17,460회 | **규민 개량 (`pc=0.6`)** | **1.941 ± 0.116** | **1.689 (`seed=8`)** | **23/30** |
| 코드 설계대로 | 규민 원본 (`pc=0.9`) | 1.993 ± 0.116 | 1.824 (`seed=3`) | 20/30 |
| 코드 설계대로 | **규민 개량 (`pc=0.6`)** | **1.941 ± 0.131** | **1.689 (`seed=8`)** | **23/30** |

### 🚀 사용 방법
1. **Cell 2**를 실행해 GitHub 저장소에서 최신 코드를 동기화(또는 내장 폴백 로드)합니다.
2. **Cell 3**을 실행해 8종 완료 판정 테스트 통과 여부를 확인합니다.
3. **Cell 4**에서 버전·실행 모드·시드 등을 선택해 GA 최적화를 실행합니다.
4. **Cell 5~6**에서 교수님 템플릿 양식(구장별 대진표 & 이동거리표)과 수렴 곡선을 확인합니다.

In [ ]:
#@title 🔗 1. GitHub 저장소 연동 및 소스 파일 동기화 { display-mode: "form" }
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Gyumin225/OR2-project.git"
REPO_NAME = "OR2-project"

_FALLBACK_IMPROVED = "\"\"\"\nO.R.2 프로야구(1982 KBO 6개 구단) 10일 경기 일정 최적화 — 통합 최적안\n====================================================================\n각 설계안에서 검증된 최적 요소만 통합한 최종 GA 코드 (Python 3 표준 라이브러리 전용)\n\n[채택 구성 및 출처]\n1. 염색체 표현 및 제약 처리 (유진안 + 호연안):\n   - 원형법(Circle Method) 기반 특수 인코딩 x = (sigma, pi, h) (유전자 31개)\n     * sigma: 길이 6 순열 (0..5) — 자리 p에 배정된 구단\n     * pi   : 길이 10 순열 (0..9) — d일차에 사용할 기본 매칭 사본 (0..4=사본 A, 5..9=사본 B)\n     * h    : 길이 15 이진 비트 (0/1) — 각 쌍 n의 사본 A 홈팀 결정 (사본 B는 자동 반대)\n   - 디코딩 규칙 자체가 두 하드 제약((1) 매일 모든 팀 1경기, (2) 각 쌍 홈1·원정1)을\n     100% 보장하므로 페널티 계수 튜닝이나 백트래킹 수리 단계가 불필요함 (교안 29쪽 1순위)\n\n2. 적합도 함수 (OR2_적합도함수_v2.md + 진우안):\n   - 7개 공정성 지표(G1~G7) + 두 기준점 정규화(G_star, G0) + 가중치 w=(1,1,1,1,1,1,3)\n   - 정규화 비용 C(x)가 음수가 될 수 있으므로 토너먼트 선택에서는 C(x)를 직접 최소화하고,\n     f(x) = 1 / (1 + C(x))는 보고서·그래프용 보조 지표로만 계산 (진우안)\n\n3. GA 연산자 및 세대 교체 (유진안 + 규민안):\n   - 초기 개체군: 무작위 (sigma, pi, h) N=60개 (디코딩 일정 기준 중복 제거 + 독립 검증)\n   - 선택: 토너먼트 선택 (k=3, 비복원 k개 추출 후 최소 C(x) 선택, 교안 39~41쪽)\n   - 교차: pc=0.6 (규민 2장 설계값; 4장 실험용으로 0.0, 0.5, 0.9 설정 가능)\n     * sigma: 부모 A 상속 / pi: 순서 교차(OX, 교안 50쪽) / h: 1점 교차(교안 44쪽)\n   - 변이: 염색체당 확률 pm=0.3, 실행 가능성 보존 4종 연산자 균등 선택 (교안 52~54쪽)\n     * (a) 일자 교환 (pi swap) / (b) 구간 역전 (pi inversion)\n     * (c) 홈/원정 비트 뒤집기 (h bit-flip) / (d) 구단 교환 (sigma swap)\n   - 세대 교체: 엘리트 보존(e=2, 교안 57쪽) + 디코딩 일정 기준 중복 제거(조기 수렴 방지)\n   - 종료 조건: 최대 세대(G_max=500, 약 29,060회 평가) 또는 정체(stall=150세대) (규민안)\n   - 다양성 추적: 매 세대 디코딩 기준 서로 다른 일정 수(distinct_schedules) 기록 (규민안)\n\n4. 검증 및 재현성 (진우안):\n   - 독립 제약 검증 함수 is_feasible() 및 8가지 완료 판정 테스트 내장\n   - 시드 고정 재현성, 다중 시드 탐색, 교차율/가중치 민감도 실험 모드 지원\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport statistics\nimport sys\nimport time\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nif hasattr(sys.stdout, \"reconfigure\"):\n    sys.stdout.reconfigure(encoding=\"utf-8\")\n\n# ---------------------------------------------------------------------------\n# 1. 문제 상수 및 거리 행렬 (과제 안내서 Table 1 / OR2_적합도함수_v2.md)\n# ---------------------------------------------------------------------------\nTEAMS: Tuple[str, ...] = (\"MBC\", \"삼미\", \"OB\", \"해태\", \"삼성\", \"롯데\")\nN_TEAMS: int = 6\nN_DAYS: int = 10\n\n# 구단 인덱스: 0=MBC(서울), 1=삼미(인천), 2=OB(대전), 3=해태(광주), 4=삼성(대구), 5=롯데(부산)\nDIST: Tuple[Tuple[int, ...], ...] = (\n    (0,   35,  165, 330, 300, 400),  # MBC (서울)  -> B_t = 2,460\n    (35,  0,   180, 320, 320, 420),  # 삼미 (인천) -> B_t = 2,550\n    (165, 180, 0,   170, 150, 250),  # OB (대전)   -> B_t = 1,830\n    (330, 320, 170, 0,   210, 270),  # 해태 (광주) -> B_t = 2,600\n    (300, 320, 150, 210, 0,   100),  # 삼성 (대구) -> B_t = 2,160\n    (400, 420, 250, 270, 100, 0),    # 롯데 (부산) -> B_t = 2,880\n)\n\n# 기준 거리 B_t = 2 * sum_{j != t} d_{tj}\nB_T: Tuple[int, ...] = tuple(2 * sum(row) for row in DIST)\n\nALL_GAMES: Tuple[Tuple[int, int], ...] = tuple(\n    (i, j) for i in range(N_TEAMS) for j in range(N_TEAMS) if i != j\n)\n\n# Circle Method 5개 기본 완전 매칭 (자리 5 고정, 0~4 회전)\nBASE_MATCHINGS: Tuple[Tuple[Tuple[int, int], ...], ...] = (\n    ((0, 5), (1, 4), (2, 3)),  # M_0\n    ((1, 5), (2, 0), (3, 4)),  # M_1\n    ((2, 5), (3, 1), (4, 0)),  # M_2\n    ((3, 5), (4, 2), (0, 1)),  # M_3\n    ((4, 5), (0, 3), (1, 2)),  # M_4\n)\n\n# 적합도 v2 기준값(G0: 무작위 1,000개 평균), 이상값(G_STAR), 가중치 프리셋\nG0_DEFAULT: Tuple[float, ...] = (11611.0, 2379.0, 0.112, 1229.0, 4.44, 1.88, 3.15)\nG_STAR_DEFAULT: Tuple[float, ...] = (7620.0, 1485.0, 0.005, 840.0, 1.0, 0.0, 0.0)\nWEIGHT_PRESETS: Dict[str, Tuple[float, ...]] = {\n    \"base\": (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0),        # 기준\n    \"fairness\": (1.0, 2.0, 2.0, 2.0, 2.0, 1.0, 3.0),    # 공정성 중시\n    \"efficiency\": (2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0),  # 효율 중시\n}\n\n# 타입 별칭\nGame = Tuple[int, int]              # (home_team_idx, away_team_idx)\nDaySchedule = Tuple[Game, ...]      # 하루 3경기\nSchedule = Tuple[DaySchedule, ...]  # 10일 일정\n\n\n# ---------------------------------------------------------------------------\n# 2. 염색체 표현, 디코딩, 독립 제약 검증\n# ---------------------------------------------------------------------------\n@dataclass\nclass Chromosome:\n    \"\"\"원형법 특수 인코딩 x = (sigma, pi, h)\"\"\"\n    sigma: List[int]  # 길이 6 순열 (0..5): 자리 p에 배정된 구단\n    pi: List[int]     # 길이 10 순열 (0..9): d일차 매칭 사본 (0..4=A, 5..9=B)\n    h: List[int]      # 길이 15 비트 (0/1): 쌍 n의 사본 A 홈 결정\n\n    # 디코딩 및 평가 결과 캐시\n    schedule: Optional[Schedule] = None\n    sched_key: Optional[Tuple[Tuple[Game, ...], ...]] = None\n    cost: Optional[float] = None\n    fitness: Optional[float] = None\n    g_vals: Optional[Tuple[float, ...]] = None\n\n    def invalidate(self) -> None:\n        self.schedule = None\n        self.sched_key = None\n        self.cost = None\n        self.fitness = None\n        self.g_vals = None\n\n    def clone(self) -> \"Chromosome\":\n        return Chromosome(\n            sigma=list(self.sigma),\n            pi=list(self.pi),\n            h=list(self.h),\n            schedule=self.schedule,\n            sched_key=self.sched_key,\n            cost=self.cost,\n            fitness=self.fitness,\n            g_vals=self.g_vals,\n        )\n\n\ndef decode(chrom: Chromosome) -> Schedule:\n    \"\"\"(sigma, pi, h) -> 10일 경기 일정 (각 날 3경기 (home, away))\"\"\"\n    if chrom.schedule is not None:\n        return chrom.schedule\n\n    sigma = chrom.sigma\n    pi = chrom.pi\n    h = chrom.h\n    days: List[DaySchedule] = []\n    keys: List[Tuple[Game, ...]] = []\n\n    for d in range(N_DAYS):\n        c = pi[d]\n        k = c % 5\n        is_copy_a = c < 5\n        games: List[Game] = []\n        for idx, (p, q) in enumerate(BASE_MATCHINGS[k]):\n            n = 3 * k + idx\n            home_is_p = (h[n] == 1) if is_copy_a else (h[n] == 0)\n            games.append((sigma[p], sigma[q]) if home_is_p else (sigma[q], sigma[p]))\n        day_t = tuple(games)\n        days.append(day_t)\n        keys.append(tuple(sorted(day_t)))\n\n    chrom.schedule = tuple(days)\n    chrom.sched_key = tuple(keys)\n    return chrom.schedule\n\n\ndef schedule_key(chrom: Chromosome) -> Tuple[Tuple[Game, ...], ...]:\n    \"\"\"디코딩된 일정(표현형) 기준 정규화 키 (중복 제거 및 다양성 측정용)\"\"\"\n    if chrom.sched_key is None:\n        decode(chrom)\n    assert chrom.sched_key is not None\n    return chrom.sched_key\n\n\ndef is_feasible(schedule: Sequence[Sequence[Game]]) -> bool:\n    \"\"\"독립 하드 제약 검증:\n    1) 정확히 10일\n    2) 매일 3경기이며 6개 구단이 정확히 1번씩 등장\n    3) 전체 30개 순서쌍 (홈, 원정)이 정확히 1번씩 등장\n    4) 구단별 홈 5경기, 원정 5경기 (교차 검증)\n    \"\"\"\n    if len(schedule) != N_DAYS:\n        return False\n\n    seen_games: List[Game] = []\n    home_cnt = [0] * N_TEAMS\n    away_cnt = [0] * N_TEAMS\n\n    for day in schedule:\n        if len(day) != 3:\n            return False\n        day_teams: List[int] = []\n        for home, away in day:\n            if not (0 <= home < N_TEAMS and 0 <= away < N_TEAMS) or home == away:\n                return False\n            day_teams.append(home)\n            day_teams.append(away)\n            home_cnt[home] += 1\n            away_cnt[away] += 1\n            seen_games.append((home, away))\n        if sorted(day_teams) != [0, 1, 2, 3, 4, 5]:\n            return False\n\n    if sorted(seen_games) != sorted(ALL_GAMES):\n        return False\n    if any(c != 5 for c in home_cnt) or any(c != 5 for c in away_cnt):\n        return False\n    return True\n\n\n# ---------------------------------------------------------------------------\n# 3. 적합도 함수 v2 (G1~G7, 두 기준점 정규화, C(x) 최소화)\n# ---------------------------------------------------------------------------\ndef compute_g_values(schedule: Schedule) -> Tuple[float, ...]:\n    \"\"\"일정의 7개 원시 공정성 지표 G1~G7 계산 (탐색 루프 최적화 버전)\"\"\"\n    loc = [[t] * (N_DAYS + 2) for t in range(N_TEAMS)]\n    home_flag = [[0] * N_DAYS for _ in range(N_TEAMS)]\n    opp = [[-1] * N_DAYS for _ in range(N_TEAMS)]\n\n    for d in range(N_DAYS):\n        d1 = d + 1\n        for h, a in schedule[d]:\n            loc[h][d1] = h\n            loc[a][d1] = h\n            home_flag[h][d] = 1\n            opp[h][d] = a\n            opp[a][d] = h\n\n    total_dist = 0\n    max_dist = 0\n    max_trip_all = 0\n    max_long_all = 0\n    streak_excess = 0\n    rho_list = [0.0] * N_TEAMS\n\n    for t in range(N_TEAMS):\n        lt = loc[t]\n        ht = home_flag[t]\n        steps = [DIST[lt[r]][lt[r + 1]] for r in range(N_DAYS + 1)]\n        dt = sum(steps)\n        total_dist += dt\n        if dt > max_dist:\n            max_dist = dt\n        rho_list[t] = dt / B_T[t]\n\n        # G5: 경기일 사이 9개 구간(r = 1..9) 중 300km 이상 이동 횟수\n        long_cnt = 0\n        for r in range(1, N_DAYS):\n            if steps[r] >= 300:\n                long_cnt += 1\n        if long_cnt > max_long_all:\n            max_long_all = long_cnt\n\n        # G4: 1회 원정 순회(연고지 출발 -> 연속 원정 -> 연고지 복귀) 최대 이동거리\n        d = 0\n        while d < N_DAYS:\n            if ht[d] == 0:\n                start = d\n                while d < N_DAYS and ht[d] == 0:\n                    d += 1\n                trip = sum(steps[start:d + 1])\n                if trip > max_trip_all:\n                    max_trip_all = trip\n            else:\n                d += 1\n\n        # G6: 3일 초과 연속 홈/원정 경기 수\n        run = 1\n        for d in range(1, N_DAYS):\n            if ht[d] == ht[d - 1]:\n                run += 1\n            else:\n                if run > 3:\n                    streak_excess += run - 3\n                run = 1\n        if run > 3:\n            streak_excess += run - 3\n\n    # G7: 리피터(이틀 연속 동일 상대 대진) 수\n    repeaters = 0\n    for d in range(1, N_DAYS):\n        d_prev = d - 1\n        for t in range(N_TEAMS):\n            if opp[t][d] == opp[t][d_prev]:\n                repeaters += 1\n    repeaters //= 2\n\n    # G3: 지리 보정 이동 비율 rho_t 의 모표준편차\n    rho_mean = sum(rho_list) / N_TEAMS\n    g3 = math.sqrt(sum((r - rho_mean) ** 2 for r in rho_list) / N_TEAMS)\n\n    return (\n        float(total_dist),\n        float(max_dist),\n        g3,\n        float(max_trip_all),\n        float(max_long_all),\n        float(streak_excess),\n        float(repeaters),\n    )\n\n\ndef detailed_team_stats(schedule: Schedule) -> Dict[str, object]:\n    \"\"\"보고서/리포트 출력용 구단별 상세 공정성 지표 계산\"\"\"\n    loc = [[t] * (N_DAYS + 2) for t in range(N_TEAMS)]\n    home_flag = [[0] * N_DAYS for _ in range(N_TEAMS)]\n\n    for d in range(N_DAYS):\n        for h, a in schedule[d]:\n            loc[h][d + 1] = h\n            loc[a][d + 1] = h\n            home_flag[h][d] = 1\n\n    dist_total = [0] * N_TEAMS\n    max_trip = [0] * N_TEAMS\n    long_moves = [0] * N_TEAMS\n    breaks = [0] * N_TEAMS\n    max_streak = [0] * N_TEAMS\n    patterns: List[str] = []\n\n    for t in range(N_TEAMS):\n        steps = [DIST[loc[t][r]][loc[t][r + 1]] for r in range(N_DAYS + 1)]\n        dist_total[t] = sum(steps)\n        long_moves[t] = sum(1 for r in range(1, N_DAYS) if steps[r] >= 300)\n\n        d = 0\n        while d < N_DAYS:\n            if home_flag[t][d] == 0:\n                s = d\n                while d < N_DAYS and home_flag[t][d] == 0:\n                    d += 1\n                trip = sum(steps[s:d + 1])\n                if trip > max_trip[t]:\n                    max_trip[t] = trip\n            else:\n                d += 1\n\n        run = 1\n        for d in range(1, N_DAYS + 1):\n            if d < N_DAYS and home_flag[t][d] == home_flag[t][d - 1]:\n                run += 1\n                breaks[t] += 1\n            else:\n                if run > max_streak[t]:\n                    max_streak[t] = run\n                run = 1\n\n        patterns.append(\"\".join(\"H\" if home_flag[t][d] == 1 else \"A\" for d in range(N_DAYS)))\n\n    rho = [dist_total[t] / B_T[t] for t in range(N_TEAMS)]\n    return {\n        \"D_t\": dist_total,\n        \"rho_t\": rho,\n        \"W_t\": max_trip,\n        \"L_t\": long_moves,\n        \"breaks\": breaks,\n        \"max_streak\": max_streak,\n        \"patterns\": patterns,\n    }\n\n\ndef evaluate_schedule(\n    schedule: Schedule,\n    weights: Sequence[float] = WEIGHT_PRESETS[\"base\"],\n    g0: Sequence[float] = G0_DEFAULT,\n    g_star: Sequence[float] = G_STAR_DEFAULT,\n) -> Tuple[float, float, Tuple[float, ...]]:\n    \"\"\"일정 x에 대해 (정규화 비용 C(x), 보조 적합도 f(x), 원시 지표 G1~G7) 반환\"\"\"\n    g_vals = compute_g_values(schedule)\n    cost = sum(\n        weights[k] * (g_vals[k] - g_star[k]) / (g0[k] - g_star[k])\n        for k in range(7)\n    )\n    fitness = 1.0 / (1.0 + cost) if cost > -1.0 else float(\"inf\")\n    return cost, fitness, g_vals\n\n\ndef evaluate_chrom(\n    chrom: Chromosome,\n    weights: Sequence[float] = WEIGHT_PRESETS[\"base\"],\n    g0: Sequence[float] = G0_DEFAULT,\n    g_star: Sequence[float] = G_STAR_DEFAULT,\n) -> float:\n    \"\"\"염색체 평가 후 cost 캐시 반환\"\"\"\n    if chrom.cost is None:\n        sched = decode(chrom)\n        cost, fitness, g_vals = evaluate_schedule(sched, weights, g0, g_star)\n        chrom.cost = cost\n        chrom.fitness = fitness\n        chrom.g_vals = g_vals\n    return chrom.cost\n\n\n# ---------------------------------------------------------------------------\n# 4. GA 연산자 (초기화, 선택, 교차, 변이)\n# ---------------------------------------------------------------------------\ndef random_chromosome(rng: random.Random) -> Chromosome:\n    sigma = list(range(N_TEAMS))\n    pi = list(range(N_DAYS))\n    rng.shuffle(sigma)\n    rng.shuffle(pi)\n    h = [rng.randint(0, 1) for _ in range(15)]\n    return Chromosome(sigma=sigma, pi=pi, h=h)\n\n\ndef init_population(\n    pop_size: int,\n    rng: random.Random,\n    dedup: bool = True,\n    max_attempts: int = 10000,\n) -> List[Chromosome]:\n    \"\"\"초기 개체군 생성 (유효성 검증 + 디코딩 일정 기준 중복 제거)\"\"\"\n    pop: List[Chromosome] = []\n    seen = set()\n    attempts = 0\n    while len(pop) < pop_size:\n        attempts += 1\n        if attempts > max_attempts:\n            raise RuntimeError(\"초기 개체군 중복 회피 한도 초과\")\n        ind = random_chromosome(rng)\n        sched = decode(ind)\n        assert is_feasible(sched), \"특수 인코딩 디코딩 결과가 실행 불가능합니다.\"\n        key = schedule_key(ind)\n        if dedup and key in seen:\n            continue\n        seen.add(key)\n        pop.append(ind)\n    return pop\n\n\ndef tournament_select(pop: Sequence[Chromosome], k: int, rng: random.Random) -> Chromosome:\n    \"\"\"토너먼트 선택 (k개 비복원 추출, 정규화 비용 C(x) 직접 비교)\"\"\"\n    contestants = rng.sample(pop, k)\n    return min(contestants, key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n\n\ndef order_crossover(a: Sequence[int], b: Sequence[int], rng: random.Random) -> List[int]:\n    \"\"\"순열 순서 교차 (OX, 교안 50쪽)\"\"\"\n    n = len(a)\n    i, j = sorted(rng.sample(range(n), 2))\n    child: List[Optional[int]] = [None] * n\n    child[i:j + 1] = a[i:j + 1]\n    placed = set(a[i:j + 1])\n\n    fill_positions = [(j + 1 + t) % n for t in range(n - (j - i + 1))]\n    b_seq = [b[(j + 1 + t) % n] for t in range(n) if b[(j + 1 + t) % n] not in placed]\n    for pos, val in zip(fill_positions, b_seq):\n        child[pos] = val\n    return [v for v in child if v is not None]\n\n\ndef one_point_crossover(a: Sequence[int], b: Sequence[int], rng: random.Random) -> List[int]:\n    \"\"\"이진 벡터 1점 교차 (교안 44쪽)\"\"\"\n    cut = rng.randint(1, len(a) - 1)\n    return list(a[:cut]) + list(b[cut:])\n\n\ndef crossover(p1: Chromosome, p2: Chromosome, pc: float, rng: random.Random) -> Chromosome:\n    \"\"\"부분별 교차: sigma 상속(A), pi OX, h 1점 교차 (자식 1개 생성)\"\"\"\n    if rng.random() >= pc:\n        return p1.clone()\n    return Chromosome(\n        sigma=list(p1.sigma),\n        pi=order_crossover(p1.pi, p2.pi, rng),\n        h=one_point_crossover(p1.h, p2.h, rng),\n    )\n\n\ndef mutate_once(chrom: Chromosome, rng: random.Random, weights: Optional[Sequence[float]] = None) -> None:\n    \"\"\"실행 가능성 보존 4종 변이 중 하나를 적용 (weights 없으면 균등 = 원본)\"\"\"\n    # [improved] weights: 규민 2장 설계의 연산자 비율 (a, b, c, d) = (0.4, 0.2, 0.3, 0.1)\n    op = rng.randrange(4) if weights is None else rng.choices(range(4), weights)[0]\n    if op == 0:\n        # (a) 일자 교환 (Swap days in pi)\n        i, j = rng.sample(range(N_DAYS), 2)\n        chrom.pi[i], chrom.pi[j] = chrom.pi[j], chrom.pi[i]\n    elif op == 1:\n        # (b) 구간 역전 (Inversion in pi)\n        i, j = sorted(rng.sample(range(N_DAYS), 2))\n        chrom.pi[i:j + 1] = chrom.pi[i:j + 1][::-1]\n    elif op == 2:\n        # (c) 홈/원정 비트 뒤집기 (Flip 1 bit in h -> 사본 A, B 동시 반전)\n        idx = rng.randrange(15)\n        chrom.h[idx] ^= 1\n    else:\n        # (d) 구단 자리 교환 (Swap 2 teams in sigma)\n        i, j = rng.sample(range(N_TEAMS), 2)\n        chrom.sigma[i], chrom.sigma[j] = chrom.sigma[j], chrom.sigma[i]\n    chrom.invalidate()\n\n\ndef mutate(chrom: Chromosome, pm: float, rng: random.Random, weights: Optional[Sequence[float]] = None) -> None:\n    \"\"\"염색체당 확률 pm으로 변이 적용\"\"\"\n    if rng.random() < pm:\n        mutate_once(chrom, rng, weights)\n\n\n# ---------------------------------------------------------------------------\n# 5. 메인 GA 루프 (엘리트 보존 + 중복 제거 + 정체 종료 + 다양성 추적)\n# ---------------------------------------------------------------------------\n@dataclass\nclass GAConfig:\n    pop_size: int = 60          # 개체군 크기 N\n    max_gens: int = 500         # 최대 세대 (60 + 500*58 = 29,060회 ≈ 30,000회)\n    stall_gens: int = 150       # 정체 종료 기준 세대 (0이면 최대 세대까지 고정 실행)\n    pc: float = 0.6             # [improved] 규민 2장 설계값 0.6 (원본 코드 0.9). 4장 실험: 0.0, 0.5, 0.9 비교\n    pm: float = 0.3             # 변이 확률 (염색체당)\n    tournament_k: int = 3       # 토너먼트 크기\n    elite_size: int = 2         # 엘리트 보존 수\n    dedup: bool = True          # 디코딩 일정 기준 중복 제거\n    max_dedup_muts: int = 50    # 중복 해소 최대 추가 변이 횟수\n    seed: int = 42\n    # [improved] on stall, replace this share of the non-elite population with new random schedules\n    # (random immigrants) and keep going, instead of stopping. 0 = original behaviour (stop).\n    restart_frac: float = 0.0\n    mut_weights: Optional[Tuple[float, ...]] = None   # [improved] None = uniform (original)\n    weights: Tuple[float, ...] = WEIGHT_PRESETS[\"base\"]\n    g0: Tuple[float, ...] = G0_DEFAULT\n    g_star: Tuple[float, ...] = G_STAR_DEFAULT\n\n\n@dataclass\nclass GAResult:\n    best: Chromosome\n    history: List[Tuple[int, float, float, int]] = field(default_factory=list)\n    # history 원소: (generation, best_cost, avg_cost, distinct_schedules)\n    evaluations: int = 0\n    elapsed_sec: float = 0.0\n    stopped_gen: int = 0\n\n\ndef run_ga(cfg: GAConfig) -> GAResult:\n    \"\"\"유전 알고리즘 실행\"\"\"\n    t0 = time.perf_counter()\n    rng = random.Random(cfg.seed)\n    evals = 0\n\n    pop = init_population(cfg.pop_size, rng, dedup=cfg.dedup)\n    for ind in pop:\n        evaluate_chrom(ind, cfg.weights, cfg.g0, cfg.g_star)\n        evals += 1\n\n    pop.sort(key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n    best = pop[0].clone()\n    last_improve_gen = 0\n\n    distinct_0 = len({schedule_key(ind) for ind in pop})\n    avg_0 = sum(c.cost for c in pop if c.cost is not None) / len(pop)\n    history: List[Tuple[int, float, float, int]] = [\n        (0, float(best.cost), avg_0, distinct_0)\n    ]\n\n    stopped_gen = 0\n    restarts = 0\n    for gen in range(1, cfg.max_gens + 1):\n        pop.sort(key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n\n        next_pop: List[Chromosome] = []\n        seen_keys = set()\n\n        # 엘리트 보존 (상위 e개)\n        for i in range(cfg.elite_size):\n            elite_copy = pop[i].clone()\n            next_pop.append(elite_copy)\n            seen_keys.add(schedule_key(elite_copy))\n\n        # 나머지 N - e개 자식 생성\n        while len(next_pop) < cfg.pop_size:\n            p1 = tournament_select(pop, cfg.tournament_k, rng)\n            p2 = tournament_select(pop, cfg.tournament_k, rng)\n            child = crossover(p1, p2, cfg.pc, rng)\n            mutate(child, cfg.pm, rng, cfg.mut_weights)\n\n            if cfg.dedup:\n                attempts = 0\n                while schedule_key(child) in seen_keys and attempts < cfg.max_dedup_muts:\n                    mutate_once(child, rng, cfg.mut_weights)\n                    attempts += 1\n\n            seen_keys.add(schedule_key(child))\n            evaluate_chrom(child, cfg.weights, cfg.g0, cfg.g_star)\n            evals += 1\n            next_pop.append(child)\n\n        pop = next_pop\n        cur_best = min(pop, key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n        assert cur_best.cost is not None and best.cost is not None\n        if cur_best.cost < best.cost - 1e-12:\n            best = cur_best.clone()\n            last_improve_gen = gen\n\n        distinct_cnt = len({schedule_key(ind) for ind in pop})\n        avg_cost = sum(c.cost for c in pop if c.cost is not None) / len(pop)\n        history.append((gen, float(best.cost), avg_cost, distinct_cnt))\n        stopped_gen = gen\n\n        if cfg.stall_gens > 0 and (gen - last_improve_gen) >= cfg.stall_gens:\n            if cfg.restart_frac <= 0:\n                break\n            pop.sort(key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n            n_new = int(round(cfg.restart_frac * (cfg.pop_size - cfg.elite_size)))\n            keep = pop[: cfg.pop_size - n_new]\n            keys = {schedule_key(c) for c in keep}\n            fresh: List[Chromosome] = []\n            while len(fresh) < n_new:\n                ind = random_chromosome(rng)\n                if schedule_key(ind) in keys:\n                    continue\n                keys.add(schedule_key(ind))\n                evaluate_chrom(ind, cfg.weights, cfg.g0, cfg.g_star)\n                evals += 1\n                fresh.append(ind)\n            pop = keep + fresh\n            restarts += 1\n            last_improve_gen = gen\n\n    assert is_feasible(decode(best))\n    elapsed = time.perf_counter() - t0\n    return GAResult(\n        best=best,\n        history=history,\n        evaluations=evals,\n        elapsed_sec=elapsed,\n        stopped_gen=stopped_gen,\n    )\n\n\n# ---------------------------------------------------------------------------\n# 6. 기준값 측정, 교수님 예시 대조, 리포트 포맷팅\n# ---------------------------------------------------------------------------\ndef measure_random_baseline(n_samples: int = 1000, seed: int = 0) -> Tuple[float, ...]:\n    \"\"\"무작위 (sigma, pi, h) 일정 n_samples개의 G1~G7 평균 측정\"\"\"\n    rng = random.Random(seed)\n    sums = [0.0] * 7\n    for _ in range(n_samples):\n        g_vals = compute_g_values(decode(random_chromosome(rng)))\n        for k in range(7):\n            sums[k] += g_vals[k]\n    return tuple(s / n_samples for s in sums)\n\n\ndef professor_example_schedule() -> Schedule:\n    \"\"\"예시 보고서 Table 4의 10일 경기 일정 (원정 @ 홈 -> (홈, 원정))\"\"\"\n    idx = {name: i for i, name in enumerate(TEAMS)}\n    raw = [\n        [(\"삼미\", \"MBC\"), (\"해태\", \"OB\"), (\"롯데\", \"삼성\")],  # 1일차\n        [(\"OB\", \"삼성\"), (\"해태\", \"삼미\"), (\"롯데\", \"MBC\")],  # 2일차\n        [(\"OB\", \"삼미\"), (\"삼성\", \"MBC\"), (\"롯데\", \"해태\")],  # 3일차\n        [(\"MBC\", \"삼미\"), (\"OB\", \"해태\"), (\"삼성\", \"롯데\")],  # 4일차\n        [(\"MBC\", \"삼성\"), (\"삼미\", \"OB\"), (\"해태\", \"롯데\")],  # 5일차\n        [(\"삼미\", \"삼성\"), (\"OB\", \"롯데\"), (\"해태\", \"MBC\")],  # 6일차\n        [(\"OB\", \"MBC\"), (\"삼성\", \"해태\"), (\"롯데\", \"삼미\")],  # 7일차\n        [(\"MBC\", \"해태\"), (\"삼성\", \"삼미\"), (\"롯데\", \"OB\")],  # 8일차\n        [(\"MBC\", \"롯데\"), (\"삼미\", \"해태\"), (\"삼성\", \"OB\")],  # 9일차\n        [(\"MBC\", \"OB\"), (\"삼미\", \"롯데\"), (\"해태\", \"삼성\")],  # 10일차\n    ]\n    return tuple(tuple((idx[h], idx[a]) for h, a in day) for day in raw)\n\n\ndef _disp_width(text: str) -> int:\n    \"\"\"한글(전각) 2칸, 영문/숫자(반각) 1칸 기준 출력 폭 계산\"\"\"\n    return sum(2 if unicodedata.east_asian_width(ch) in (\"W\", \"F\") else 1 for ch in text)\n\n\ndef _pad_left(text: str, width: int) -> str:\n    return text + \" \" * max(0, width - _disp_width(text))\n\n\ndef _pad_right(text: str, width: int) -> str:\n    return \" \" * max(0, width - _disp_width(text)) + text\n\n\ndef _pad_center(text: str, width: int) -> str:\n    gap = max(0, width - _disp_width(text))\n    left = gap // 2\n    return \" \" * left + text + \" \" * (gap - left)\n\n\ndef format_report(res: GAResult) -> str:\n    \"\"\"최종 10일 일정, 구단별 공정성 지표, G1~G7, 제약 검증 요약 포맷팅\"\"\"\n    best = res.best\n    sched = decode(best)\n    assert best.g_vals is not None and best.cost is not None and best.fitness is not None\n    g_vals = best.g_vals\n    stats = detailed_team_stats(sched)\n    d_t = stats[\"D_t\"]\n    rho_t = stats[\"rho_t\"]\n    w_t = stats[\"W_t\"]\n    l_t = stats[\"L_t\"]\n    breaks = stats[\"breaks\"]\n    max_streak = stats[\"max_streak\"]\n    patterns = stats[\"patterns\"]\n\n    prof_cost, _, _ = evaluate_schedule(professor_example_schedule())\n\n    lines: List[str] = []\n    lines.append(\"=\" * 82)\n    lines.append(\n        f\"최적 일정 탐색 결과 (종료 세대: {res.stopped_gen}, 총 평가: {res.evaluations:,}회, {res.elapsed_sec:.2f}s)\"\n    )\n    lines.append(\n        f\"정규화 비용 C(x) = {best.cost:.4f} (교수님 예시 {prof_cost:.4f} 대비 {best.cost - prof_cost:+.4f}) \"\n        f\"| 보조 적합도 f(x) = {best.fitness:.4f}\"\n    )\n    lines.append(f\"독립 제약 검증 is_feasible(): {is_feasible(sched)}\")\n    lines.append(\"-\" * 82)\n    lines.append(\n        f\"G1(총이동)={g_vals[0]:.0f}km | G2(최대)={g_vals[1]:.0f}km | \"\n        f\"G3(비율std)={g_vals[2]:.4f} | G4(최장원정)={g_vals[3]:.0f}km | \"\n        f\"G5(>=300km)={g_vals[4]:.0f}회 | G6(3연속초과)={g_vals[5]:.0f} | G7(리피터)={g_vals[6]:.0f}\"\n    )\n    lines.append(\"-\" * 82)\n    lines.append(\"[10일 경기 스케줄] (표기: 원정 @ 홈)\")\n    for d, day in enumerate(sched, 1):\n        day_str = \"   \".join(_pad_left(f\"{TEAMS[a]} @ {TEAMS[h]}\", 12) for h, a in day)\n        lines.append(f\"  {d:2d}일차: {day_str.rstrip()}\")\n    lines.append(\"-\" * 82)\n    lines.append(\"[구단별 공정성 지표]\")\n\n    headers = [\"구단\", \"홈/원정 패턴\", \"총이동(km)\", \"D/B비율\", \"최장원정\", \">=300km\", \"브레이크\", \"최장연속\"]\n    col_widths = [4, 12, 10, 7, 8, 7, 8, 8]\n    header_line = \"  \" + \" | \".join(_pad_center(h, w) for h, w in zip(headers, col_widths))\n    lines.append(header_line)\n    lines.append(\"  \" + \"-+-\".join(\"-\" * w for w in col_widths))\n\n    for t in range(N_TEAMS):\n        cells = [\n            _pad_left(TEAMS[t], col_widths[0]),\n            _pad_center(patterns[t], col_widths[1]),\n            _pad_right(f\"{d_t[t]:,d}\", col_widths[2]),\n            _pad_right(f\"{rho_t[t]:.3f}\", col_widths[3]),\n            _pad_right(f\"{w_t[t]:,d}\", col_widths[4]),\n            _pad_right(f\"{l_t[t]}\", col_widths[5]),\n            _pad_right(f\"{breaks[t]}\", col_widths[6]),\n            _pad_right(f\"{max_streak[t]}\", col_widths[7]),\n        ]\n        lines.append(\"  \" + \" | \".join(cells))\n\n    lines.append(\"=\" * 82)\n    return \"\\n\".join(lines)\n\n\n# ---------------------------------------------------------------------------\n# 7. 구현 완료 판정 테스트 8종 (OR2_GA_설계.md 기준)\n# ---------------------------------------------------------------------------\ndef run_completion_tests() -> None:\n    \"\"\"설계 문서의 8가지 완료 판정 테스트 실행\"\"\"\n    rng = random.Random(12345)\n\n    # Test 1: 염색체 및 카드 무결성 (초기화, OX/1점 교차, 4종 변이)\n    for _ in range(100):\n        p1 = random_chromosome(rng)\n        p2 = random_chromosome(rng)\n        c = crossover(p1, p2, 1.0, rng)\n        mutate(c, 1.0, rng)\n        assert sorted(c.sigma) == [0, 1, 2, 3, 4, 5]\n        assert sorted(c.pi) == list(range(10))\n        assert len(c.h) == 15 and all(b in (0, 1) for b in c.h)\n        assert is_feasible(decode(c))\n\n    # Test 2 & 3: 무작위 염색체 1,000개 실행 가능성 및 표현형 다양성 비율\n    decoded_keys = set()\n    for _ in range(1000):\n        ind = random_chromosome(rng)\n        sched = decode(ind)\n        assert is_feasible(sched)\n        decoded_keys.add(schedule_key(ind))\n    unique_ratio = len(decoded_keys) / 1000.0\n    assert unique_ratio > 0.99, f\"다양성 비율이 지나치게 낮음: {unique_ratio}\"\n\n    # Test 4: 비용 계산 재현성 및 교수님 예시 일정 대조\n    prof_sched = professor_example_schedule()\n    assert is_feasible(prof_sched)\n    c1, f1, g1 = evaluate_schedule(prof_sched)\n    c2, f2, _ = evaluate_schedule(prof_sched)\n    assert math.isclose(c1, c2, rel_tol=0.0, abs_tol=1e-12)\n    assert math.isclose(f1, f2, rel_tol=0.0, abs_tol=1e-12)\n    assert g1[0] == 9280.0 and g1[1] == 1585.0\n    assert abs(c1 - 2.013) < 0.01, f\"교수님 예시 v2 비용 불일치: {c1}\"\n\n    # Test 5 & 6 & 7: 세대 크기(60), 엘리트 단조성, 동일 시드 재현성\n    res1 = run_ga(GAConfig(pop_size=60, max_gens=50, stall_gens=0, seed=42))\n    res2 = run_ga(GAConfig(pop_size=60, max_gens=50, stall_gens=0, seed=42))\n    assert len(res1.history) == 51\n    for gen in range(1, len(res1.history)):\n        assert res1.history[gen][1] <= res1.history[gen - 1][1] + 1e-12\n    assert res1.best.cost == res2.best.cost\n    assert decode(res1.best) == decode(res2.best)\n\n    # Test 8: 최종 출력 생성 검증\n    rep = format_report(res1)\n    assert \"10일 경기 스케줄\" in rep and \"True\" in rep\n\n    print(\n        f\"[완료 판정 테스트 8/8 통과] \"\n        f\"무작위 1,000개 실행가능 100% (고유 일정 {len(decoded_keys)}/1000={unique_ratio:.3f}), \"\n        f\"교수님 예시 C(x)={c1:.4f}\"\n    )\n\n\n# ---------------------------------------------------------------------------\n# 8. 실험 실행기 (다중 시드 / 교차율 비교 / 가중치 민감도)\n# ---------------------------------------------------------------------------\ndef run_multi_seed(\n    seeds: int = 8,\n    pc: float = 0.6,\n    pm: float = 0.3,\n    pop_size: int = 60,\n    max_gens: int = 500,\n    stall_gens: int = 150,\n    dedup: bool = True,\n    weights: Tuple[float, ...] = WEIGHT_PRESETS[\"base\"],\n    label: str = \"기본 설정\",\n) -> GAResult:\n    \"\"\"다중 시드 실행 후 통계 및 최선 해 반환\"\"\"\n    costs: List[float] = []\n    distincts: List[int] = []\n    best_res: Optional[GAResult] = None\n    t0 = time.perf_counter()\n\n    for s in range(seeds):\n        cfg = GAConfig(\n            pop_size=pop_size,\n            max_gens=max_gens,\n            stall_gens=stall_gens,\n            pc=pc,\n            pm=pm,\n            tournament_k=3,\n            elite_size=2,\n            dedup=dedup,\n            seed=s,\n            weights=weights,\n        )\n        res = run_ga(cfg)\n        assert res.best.cost is not None\n        costs.append(res.best.cost)\n        distincts.append(res.history[-1][3])\n        if best_res is None or res.best.cost < best_res.best.cost:  # type: ignore[operator]\n            best_res = res\n\n    assert best_res is not None\n    mean_c = statistics.mean(costs)\n    std_c = statistics.stdev(costs) if len(costs) > 1 else 0.0\n    print(\n        f\"[{label} | seeds={seeds}, pc={pc}, pm={pm}, N={pop_size}, G={max_gens}, stall={stall_gens}] \"\n        f\"평균={mean_c:.3f} ± {std_c:.3f} | 최선={min(costs):.3f} | 최악={max(costs):.3f} | \"\n        f\"고유일정={statistics.mean(distincts):.1f}/{pop_size} | 시간={time.perf_counter()-t0:.1f}s\"\n    )\n    return best_res\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"O.R.2 프로야구 10일 일정 GA (통합 최적안)\")\n    parser.add_argument(\"--mode\", choices=[\"demo\", \"single\", \"ablation\"], default=\"demo\")\n    parser.add_argument(\"--seeds\", type=int, default=8)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--pc\", type=float, default=0.6)  # [improved]\n    parser.add_argument(\"--pm\", type=float, default=0.3)\n    parser.add_argument(\"--pop-size\", type=int, default=60)\n    parser.add_argument(\"--max-gens\", type=int, default=500)\n    parser.add_argument(\"--stall\", type=int, default=150)\n    parser.add_argument(\"--weight\", choices=list(WEIGHT_PRESETS.keys()), default=\"base\")\n    args, _ = parser.parse_known_args()\n\n    run_completion_tests()\n\n    if args.mode == \"single\":\n        res = run_ga(\n            GAConfig(\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=args.stall,\n                pc=args.pc,\n                pm=args.pm,\n                seed=args.seed,\n                weights=WEIGHT_PRESETS[args.weight],\n            )\n        )\n        print(format_report(res))\n    elif args.mode == \"ablation\":\n        print(\"\\n--- 교차 확률 비교 (4장 보고용: pc = 0.0, 0.5, 0.9) ---\")\n        for pc in (0.0, 0.5, 0.9):\n            run_multi_seed(\n                seeds=args.seeds,\n                pc=pc,\n                pm=args.pm,\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=0,\n                label=f\"pc={pc}\",\n            )\n        print(\"\\n--- 가중치 민감도 비교 (base / fairness / efficiency) ---\")\n        for name, w in WEIGHT_PRESETS.items():\n            best_w = run_multi_seed(\n                seeds=args.seeds,\n                pc=args.pc,\n                pm=args.pm,\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=args.stall,\n                weights=w,\n                label=f\"weight={name}\",\n            )\n            assert best_w.best.g_vals is not None\n            print(f\"  -> {name} 최선 G1~G7: {[round(x, 4) for x in best_w.best.g_vals]}\")\n    else:\n        print(\"\\n--- 통합 최적안 다중 시드 탐색 ---\")\n        best_res = run_multi_seed(\n            seeds=args.seeds,\n            pc=args.pc,\n            pm=args.pm,\n            pop_size=args.pop_size,\n            max_gens=args.max_gens,\n            stall_gens=args.stall,\n            weights=WEIGHT_PRESETS[args.weight],\n            label=\"채택안(유진+규민+진우 통합)\",\n        )\n        print(format_report(best_res))\n\n\nif __name__ == \"__main__\":\n    main()\n"
_FALLBACK_ORIGINAL = "\"\"\"\nO.R.2 프로야구(1982 KBO 6개 구단) 10일 경기 일정 최적화 — 통합 최적안\n====================================================================\n각 설계안에서 검증된 최적 요소만 통합한 최종 GA 코드 (Python 3 표준 라이브러리 전용)\n\n[채택 구성 및 출처]\n1. 염색체 표현 및 제약 처리 (유진안 + 호연안):\n   - 원형법(Circle Method) 기반 특수 인코딩 x = (sigma, pi, h) (유전자 31개)\n     * sigma: 길이 6 순열 (0..5) — 자리 p에 배정된 구단\n     * pi   : 길이 10 순열 (0..9) — d일차에 사용할 기본 매칭 사본 (0..4=사본 A, 5..9=사본 B)\n     * h    : 길이 15 이진 비트 (0/1) — 각 쌍 n의 사본 A 홈팀 결정 (사본 B는 자동 반대)\n   - 디코딩 규칙 자체가 두 하드 제약((1) 매일 모든 팀 1경기, (2) 각 쌍 홈1·원정1)을\n     100% 보장하므로 페널티 계수 튜닝이나 백트래킹 수리 단계가 불필요함 (교안 29쪽 1순위)\n\n2. 적합도 함수 (OR2_적합도함수_v2.md + 진우안):\n   - 7개 공정성 지표(G1~G7) + 두 기준점 정규화(G_star, G0) + 가중치 w=(1,1,1,1,1,1,3)\n   - 정규화 비용 C(x)가 음수가 될 수 있으므로 토너먼트 선택에서는 C(x)를 직접 최소화하고,\n     f(x) = 1 / (1 + C(x))는 보고서·그래프용 보조 지표로만 계산 (진우안)\n\n3. GA 연산자 및 세대 교체 (유진안 + 규민안):\n   - 초기 개체군: 무작위 (sigma, pi, h) N=60개 (디코딩 일정 기준 중복 제거 + 독립 검증)\n   - 선택: 토너먼트 선택 (k=3, 비복원 k개 추출 후 최소 C(x) 선택, 교안 39~41쪽)\n   - 교차: pc=0.9 (4장 실험용으로 0.0, 0.5, 0.9 설정 가능)\n     * sigma: 부모 A 상속 / pi: 순서 교차(OX, 교안 50쪽) / h: 1점 교차(교안 44쪽)\n   - 변이: 염색체당 확률 pm=0.3, 실행 가능성 보존 4종 연산자 균등 선택 (교안 52~54쪽)\n     * (a) 일자 교환 (pi swap) / (b) 구간 역전 (pi inversion)\n     * (c) 홈/원정 비트 뒤집기 (h bit-flip) / (d) 구단 교환 (sigma swap)\n   - 세대 교체: 엘리트 보존(e=2, 교안 57쪽) + 디코딩 일정 기준 중복 제거(조기 수렴 방지)\n   - 종료 조건: 최대 세대(G_max=500, 약 29,060회 평가) 또는 정체(stall=150세대) (규민안)\n   - 다양성 추적: 매 세대 디코딩 기준 서로 다른 일정 수(distinct_schedules) 기록 (규민안)\n\n4. 검증 및 재현성 (진우안):\n   - 독립 제약 검증 함수 is_feasible() 및 8가지 완료 판정 테스트 내장\n   - 시드 고정 재현성, 다중 시드 탐색, 교차율/가중치 민감도 실험 모드 지원\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport statistics\nimport sys\nimport time\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nif hasattr(sys.stdout, \"reconfigure\"):\n    sys.stdout.reconfigure(encoding=\"utf-8\")\n\n# ---------------------------------------------------------------------------\n# 1. 문제 상수 및 거리 행렬 (과제 안내서 Table 1 / OR2_적합도함수_v2.md)\n# ---------------------------------------------------------------------------\nTEAMS: Tuple[str, ...] = (\"MBC\", \"삼미\", \"OB\", \"해태\", \"삼성\", \"롯데\")\nN_TEAMS: int = 6\nN_DAYS: int = 10\n\n# 구단 인덱스: 0=MBC(서울), 1=삼미(인천), 2=OB(대전), 3=해태(광주), 4=삼성(대구), 5=롯데(부산)\nDIST: Tuple[Tuple[int, ...], ...] = (\n    (0,   35,  165, 330, 300, 400),  # MBC (서울)  -> B_t = 2,460\n    (35,  0,   180, 320, 320, 420),  # 삼미 (인천) -> B_t = 2,550\n    (165, 180, 0,   170, 150, 250),  # OB (대전)   -> B_t = 1,830\n    (330, 320, 170, 0,   210, 270),  # 해태 (광주) -> B_t = 2,600\n    (300, 320, 150, 210, 0,   100),  # 삼성 (대구) -> B_t = 2,160\n    (400, 420, 250, 270, 100, 0),    # 롯데 (부산) -> B_t = 2,880\n)\n\n# 기준 거리 B_t = 2 * sum_{j != t} d_{tj}\nB_T: Tuple[int, ...] = tuple(2 * sum(row) for row in DIST)\n\nALL_GAMES: Tuple[Tuple[int, int], ...] = tuple(\n    (i, j) for i in range(N_TEAMS) for j in range(N_TEAMS) if i != j\n)\n\n# Circle Method 5개 기본 완전 매칭 (자리 5 고정, 0~4 회전)\nBASE_MATCHINGS: Tuple[Tuple[Tuple[int, int], ...], ...] = (\n    ((0, 5), (1, 4), (2, 3)),  # M_0\n    ((1, 5), (2, 0), (3, 4)),  # M_1\n    ((2, 5), (3, 1), (4, 0)),  # M_2\n    ((3, 5), (4, 2), (0, 1)),  # M_3\n    ((4, 5), (0, 3), (1, 2)),  # M_4\n)\n\n# 적합도 v2 기준값(G0: 무작위 1,000개 평균), 이상값(G_STAR), 가중치 프리셋\nG0_DEFAULT: Tuple[float, ...] = (11611.0, 2379.0, 0.112, 1229.0, 4.44, 1.88, 3.15)\nG_STAR_DEFAULT: Tuple[float, ...] = (7620.0, 1485.0, 0.005, 840.0, 1.0, 0.0, 0.0)\nWEIGHT_PRESETS: Dict[str, Tuple[float, ...]] = {\n    \"base\": (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0),        # 기준\n    \"fairness\": (1.0, 2.0, 2.0, 2.0, 2.0, 1.0, 3.0),    # 공정성 중시\n    \"efficiency\": (2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0),  # 효율 중시\n}\n\n# 타입 별칭\nGame = Tuple[int, int]              # (home_team_idx, away_team_idx)\nDaySchedule = Tuple[Game, ...]      # 하루 3경기\nSchedule = Tuple[DaySchedule, ...]  # 10일 일정\n\n\n# ---------------------------------------------------------------------------\n# 2. 염색체 표현, 디코딩, 독립 제약 검증\n# ---------------------------------------------------------------------------\n@dataclass\nclass Chromosome:\n    \"\"\"원형법 특수 인코딩 x = (sigma, pi, h)\"\"\"\n    sigma: List[int]  # 길이 6 순열 (0..5): 자리 p에 배정된 구단\n    pi: List[int]     # 길이 10 순열 (0..9): d일차 매칭 사본 (0..4=A, 5..9=B)\n    h: List[int]      # 길이 15 비트 (0/1): 쌍 n의 사본 A 홈 결정\n\n    # 디코딩 및 평가 결과 캐시\n    schedule: Optional[Schedule] = None\n    sched_key: Optional[Tuple[Tuple[Game, ...], ...]] = None\n    cost: Optional[float] = None\n    fitness: Optional[float] = None\n    g_vals: Optional[Tuple[float, ...]] = None\n\n    def invalidate(self) -> None:\n        self.schedule = None\n        self.sched_key = None\n        self.cost = None\n        self.fitness = None\n        self.g_vals = None\n\n    def clone(self) -> \"Chromosome\":\n        return Chromosome(\n            sigma=list(self.sigma),\n            pi=list(self.pi),\n            h=list(self.h),\n            schedule=self.schedule,\n            sched_key=self.sched_key,\n            cost=self.cost,\n            fitness=self.fitness,\n            g_vals=self.g_vals,\n        )\n\n\ndef decode(chrom: Chromosome) -> Schedule:\n    \"\"\"(sigma, pi, h) -> 10일 경기 일정 (각 날 3경기 (home, away))\"\"\"\n    if chrom.schedule is not None:\n        return chrom.schedule\n\n    sigma = chrom.sigma\n    pi = chrom.pi\n    h = chrom.h\n    days: List[DaySchedule] = []\n    keys: List[Tuple[Game, ...]] = []\n\n    for d in range(N_DAYS):\n        c = pi[d]\n        k = c % 5\n        is_copy_a = c < 5\n        games: List[Game] = []\n        for idx, (p, q) in enumerate(BASE_MATCHINGS[k]):\n            n = 3 * k + idx\n            home_is_p = (h[n] == 1) if is_copy_a else (h[n] == 0)\n            games.append((sigma[p], sigma[q]) if home_is_p else (sigma[q], sigma[p]))\n        day_t = tuple(games)\n        days.append(day_t)\n        keys.append(tuple(sorted(day_t)))\n\n    chrom.schedule = tuple(days)\n    chrom.sched_key = tuple(keys)\n    return chrom.schedule\n\n\ndef schedule_key(chrom: Chromosome) -> Tuple[Tuple[Game, ...], ...]:\n    \"\"\"디코딩된 일정(표현형) 기준 정규화 키 (중복 제거 및 다양성 측정용)\"\"\"\n    if chrom.sched_key is None:\n        decode(chrom)\n    assert chrom.sched_key is not None\n    return chrom.sched_key\n\n\ndef is_feasible(schedule: Sequence[Sequence[Game]]) -> bool:\n    \"\"\"독립 하드 제약 검증:\n    1) 정확히 10일\n    2) 매일 3경기이며 6개 구단이 정확히 1번씩 등장\n    3) 전체 30개 순서쌍 (홈, 원정)이 정확히 1번씩 등장\n    4) 구단별 홈 5경기, 원정 5경기 (교차 검증)\n    \"\"\"\n    if len(schedule) != N_DAYS:\n        return False\n\n    seen_games: List[Game] = []\n    home_cnt = [0] * N_TEAMS\n    away_cnt = [0] * N_TEAMS\n\n    for day in schedule:\n        if len(day) != 3:\n            return False\n        day_teams: List[int] = []\n        for home, away in day:\n            if not (0 <= home < N_TEAMS and 0 <= away < N_TEAMS) or home == away:\n                return False\n            day_teams.append(home)\n            day_teams.append(away)\n            home_cnt[home] += 1\n            away_cnt[away] += 1\n            seen_games.append((home, away))\n        if sorted(day_teams) != [0, 1, 2, 3, 4, 5]:\n            return False\n\n    if sorted(seen_games) != sorted(ALL_GAMES):\n        return False\n    if any(c != 5 for c in home_cnt) or any(c != 5 for c in away_cnt):\n        return False\n    return True\n\n\n# ---------------------------------------------------------------------------\n# 3. 적합도 함수 v2 (G1~G7, 두 기준점 정규화, C(x) 최소화)\n# ---------------------------------------------------------------------------\ndef compute_g_values(schedule: Schedule) -> Tuple[float, ...]:\n    \"\"\"일정의 7개 원시 공정성 지표 G1~G7 계산 (탐색 루프 최적화 버전)\"\"\"\n    loc = [[t] * (N_DAYS + 2) for t in range(N_TEAMS)]\n    home_flag = [[0] * N_DAYS for _ in range(N_TEAMS)]\n    opp = [[-1] * N_DAYS for _ in range(N_TEAMS)]\n\n    for d in range(N_DAYS):\n        d1 = d + 1\n        for h, a in schedule[d]:\n            loc[h][d1] = h\n            loc[a][d1] = h\n            home_flag[h][d] = 1\n            opp[h][d] = a\n            opp[a][d] = h\n\n    total_dist = 0\n    max_dist = 0\n    max_trip_all = 0\n    max_long_all = 0\n    streak_excess = 0\n    rho_list = [0.0] * N_TEAMS\n\n    for t in range(N_TEAMS):\n        lt = loc[t]\n        ht = home_flag[t]\n        steps = [DIST[lt[r]][lt[r + 1]] for r in range(N_DAYS + 1)]\n        dt = sum(steps)\n        total_dist += dt\n        if dt > max_dist:\n            max_dist = dt\n        rho_list[t] = dt / B_T[t]\n\n        # G5: 경기일 사이 9개 구간(r = 1..9) 중 300km 이상 이동 횟수\n        long_cnt = 0\n        for r in range(1, N_DAYS):\n            if steps[r] >= 300:\n                long_cnt += 1\n        if long_cnt > max_long_all:\n            max_long_all = long_cnt\n\n        # G4: 1회 원정 순회(연고지 출발 -> 연속 원정 -> 연고지 복귀) 최대 이동거리\n        d = 0\n        while d < N_DAYS:\n            if ht[d] == 0:\n                start = d\n                while d < N_DAYS and ht[d] == 0:\n                    d += 1\n                trip = sum(steps[start:d + 1])\n                if trip > max_trip_all:\n                    max_trip_all = trip\n            else:\n                d += 1\n\n        # G6: 3일 초과 연속 홈/원정 경기 수\n        run = 1\n        for d in range(1, N_DAYS):\n            if ht[d] == ht[d - 1]:\n                run += 1\n            else:\n                if run > 3:\n                    streak_excess += run - 3\n                run = 1\n        if run > 3:\n            streak_excess += run - 3\n\n    # G7: 리피터(이틀 연속 동일 상대 대진) 수\n    repeaters = 0\n    for d in range(1, N_DAYS):\n        d_prev = d - 1\n        for t in range(N_TEAMS):\n            if opp[t][d] == opp[t][d_prev]:\n                repeaters += 1\n    repeaters //= 2\n\n    # G3: 지리 보정 이동 비율 rho_t 의 모표준편차\n    rho_mean = sum(rho_list) / N_TEAMS\n    g3 = math.sqrt(sum((r - rho_mean) ** 2 for r in rho_list) / N_TEAMS)\n\n    return (\n        float(total_dist),\n        float(max_dist),\n        g3,\n        float(max_trip_all),\n        float(max_long_all),\n        float(streak_excess),\n        float(repeaters),\n    )\n\n\ndef detailed_team_stats(schedule: Schedule) -> Dict[str, object]:\n    \"\"\"보고서/리포트 출력용 구단별 상세 공정성 지표 계산\"\"\"\n    loc = [[t] * (N_DAYS + 2) for t in range(N_TEAMS)]\n    home_flag = [[0] * N_DAYS for _ in range(N_TEAMS)]\n\n    for d in range(N_DAYS):\n        for h, a in schedule[d]:\n            loc[h][d + 1] = h\n            loc[a][d + 1] = h\n            home_flag[h][d] = 1\n\n    dist_total = [0] * N_TEAMS\n    max_trip = [0] * N_TEAMS\n    long_moves = [0] * N_TEAMS\n    breaks = [0] * N_TEAMS\n    max_streak = [0] * N_TEAMS\n    patterns: List[str] = []\n\n    for t in range(N_TEAMS):\n        steps = [DIST[loc[t][r]][loc[t][r + 1]] for r in range(N_DAYS + 1)]\n        dist_total[t] = sum(steps)\n        long_moves[t] = sum(1 for r in range(1, N_DAYS) if steps[r] >= 300)\n\n        d = 0\n        while d < N_DAYS:\n            if home_flag[t][d] == 0:\n                s = d\n                while d < N_DAYS and home_flag[t][d] == 0:\n                    d += 1\n                trip = sum(steps[s:d + 1])\n                if trip > max_trip[t]:\n                    max_trip[t] = trip\n            else:\n                d += 1\n\n        run = 1\n        for d in range(1, N_DAYS + 1):\n            if d < N_DAYS and home_flag[t][d] == home_flag[t][d - 1]:\n                run += 1\n                breaks[t] += 1\n            else:\n                if run > max_streak[t]:\n                    max_streak[t] = run\n                run = 1\n\n        patterns.append(\"\".join(\"H\" if home_flag[t][d] == 1 else \"A\" for d in range(N_DAYS)))\n\n    rho = [dist_total[t] / B_T[t] for t in range(N_TEAMS)]\n    return {\n        \"D_t\": dist_total,\n        \"rho_t\": rho,\n        \"W_t\": max_trip,\n        \"L_t\": long_moves,\n        \"breaks\": breaks,\n        \"max_streak\": max_streak,\n        \"patterns\": patterns,\n    }\n\n\ndef evaluate_schedule(\n    schedule: Schedule,\n    weights: Sequence[float] = WEIGHT_PRESETS[\"base\"],\n    g0: Sequence[float] = G0_DEFAULT,\n    g_star: Sequence[float] = G_STAR_DEFAULT,\n) -> Tuple[float, float, Tuple[float, ...]]:\n    \"\"\"일정 x에 대해 (정규화 비용 C(x), 보조 적합도 f(x), 원시 지표 G1~G7) 반환\"\"\"\n    g_vals = compute_g_values(schedule)\n    cost = sum(\n        weights[k] * (g_vals[k] - g_star[k]) / (g0[k] - g_star[k])\n        for k in range(7)\n    )\n    fitness = 1.0 / (1.0 + cost) if cost > -1.0 else float(\"inf\")\n    return cost, fitness, g_vals\n\n\ndef evaluate_chrom(\n    chrom: Chromosome,\n    weights: Sequence[float] = WEIGHT_PRESETS[\"base\"],\n    g0: Sequence[float] = G0_DEFAULT,\n    g_star: Sequence[float] = G_STAR_DEFAULT,\n) -> float:\n    \"\"\"염색체 평가 후 cost 캐시 반환\"\"\"\n    if chrom.cost is None:\n        sched = decode(chrom)\n        cost, fitness, g_vals = evaluate_schedule(sched, weights, g0, g_star)\n        chrom.cost = cost\n        chrom.fitness = fitness\n        chrom.g_vals = g_vals\n    return chrom.cost\n\n\n# ---------------------------------------------------------------------------\n# 4. GA 연산자 (초기화, 선택, 교차, 변이)\n# ---------------------------------------------------------------------------\ndef random_chromosome(rng: random.Random) -> Chromosome:\n    sigma = list(range(N_TEAMS))\n    pi = list(range(N_DAYS))\n    rng.shuffle(sigma)\n    rng.shuffle(pi)\n    h = [rng.randint(0, 1) for _ in range(15)]\n    return Chromosome(sigma=sigma, pi=pi, h=h)\n\n\ndef init_population(\n    pop_size: int,\n    rng: random.Random,\n    dedup: bool = True,\n    max_attempts: int = 10000,\n) -> List[Chromosome]:\n    \"\"\"초기 개체군 생성 (유효성 검증 + 디코딩 일정 기준 중복 제거)\"\"\"\n    pop: List[Chromosome] = []\n    seen = set()\n    attempts = 0\n    while len(pop) < pop_size:\n        attempts += 1\n        if attempts > max_attempts:\n            raise RuntimeError(\"초기 개체군 중복 회피 한도 초과\")\n        ind = random_chromosome(rng)\n        sched = decode(ind)\n        assert is_feasible(sched), \"특수 인코딩 디코딩 결과가 실행 불가능합니다.\"\n        key = schedule_key(ind)\n        if dedup and key in seen:\n            continue\n        seen.add(key)\n        pop.append(ind)\n    return pop\n\n\ndef tournament_select(pop: Sequence[Chromosome], k: int, rng: random.Random) -> Chromosome:\n    \"\"\"토너먼트 선택 (k개 비복원 추출, 정규화 비용 C(x) 직접 비교)\"\"\"\n    contestants = rng.sample(pop, k)\n    return min(contestants, key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n\n\ndef order_crossover(a: Sequence[int], b: Sequence[int], rng: random.Random) -> List[int]:\n    \"\"\"순열 순서 교차 (OX, 교안 50쪽)\"\"\"\n    n = len(a)\n    i, j = sorted(rng.sample(range(n), 2))\n    child: List[Optional[int]] = [None] * n\n    child[i:j + 1] = a[i:j + 1]\n    placed = set(a[i:j + 1])\n\n    fill_positions = [(j + 1 + t) % n for t in range(n - (j - i + 1))]\n    b_seq = [b[(j + 1 + t) % n] for t in range(n) if b[(j + 1 + t) % n] not in placed]\n    for pos, val in zip(fill_positions, b_seq):\n        child[pos] = val\n    return [v for v in child if v is not None]\n\n\ndef one_point_crossover(a: Sequence[int], b: Sequence[int], rng: random.Random) -> List[int]:\n    \"\"\"이진 벡터 1점 교차 (교안 44쪽)\"\"\"\n    cut = rng.randint(1, len(a) - 1)\n    return list(a[:cut]) + list(b[cut:])\n\n\ndef crossover(p1: Chromosome, p2: Chromosome, pc: float, rng: random.Random) -> Chromosome:\n    \"\"\"부분별 교차: sigma 상속(A), pi OX, h 1점 교차 (자식 1개 생성)\"\"\"\n    if rng.random() >= pc:\n        return p1.clone()\n    return Chromosome(\n        sigma=list(p1.sigma),\n        pi=order_crossover(p1.pi, p2.pi, rng),\n        h=one_point_crossover(p1.h, p2.h, rng),\n    )\n\n\ndef mutate_once(chrom: Chromosome, rng: random.Random) -> None:\n    \"\"\"실행 가능성 보존 4종 변이 중 하나를 균등 확률로 적용\"\"\"\n    op = rng.randrange(4)\n    if op == 0:\n        # (a) 일자 교환 (Swap days in pi)\n        i, j = rng.sample(range(N_DAYS), 2)\n        chrom.pi[i], chrom.pi[j] = chrom.pi[j], chrom.pi[i]\n    elif op == 1:\n        # (b) 구간 역전 (Inversion in pi)\n        i, j = sorted(rng.sample(range(N_DAYS), 2))\n        chrom.pi[i:j + 1] = chrom.pi[i:j + 1][::-1]\n    elif op == 2:\n        # (c) 홈/원정 비트 뒤집기 (Flip 1 bit in h -> 사본 A, B 동시 반전)\n        idx = rng.randrange(15)\n        chrom.h[idx] ^= 1\n    else:\n        # (d) 구단 자리 교환 (Swap 2 teams in sigma)\n        i, j = rng.sample(range(N_TEAMS), 2)\n        chrom.sigma[i], chrom.sigma[j] = chrom.sigma[j], chrom.sigma[i]\n    chrom.invalidate()\n\n\ndef mutate(chrom: Chromosome, pm: float, rng: random.Random) -> None:\n    \"\"\"염색체당 확률 pm으로 변이 적용\"\"\"\n    if rng.random() < pm:\n        mutate_once(chrom, rng)\n\n\n# ---------------------------------------------------------------------------\n# 5. 메인 GA 루프 (엘리트 보존 + 중복 제거 + 정체 종료 + 다양성 추적)\n# ---------------------------------------------------------------------------\n@dataclass\nclass GAConfig:\n    pop_size: int = 60          # 개체군 크기 N\n    max_gens: int = 500         # 최대 세대 (60 + 500*58 = 29,060회 ≈ 30,000회)\n    stall_gens: int = 150       # 정체 종료 기준 세대 (0이면 최대 세대까지 고정 실행)\n    pc: float = 0.9             # 교차 확률 (4장 실험: 0.0, 0.5, 0.9 비교)\n    pm: float = 0.3             # 변이 확률 (염색체당)\n    tournament_k: int = 3       # 토너먼트 크기\n    elite_size: int = 2         # 엘리트 보존 수\n    dedup: bool = True          # 디코딩 일정 기준 중복 제거\n    max_dedup_muts: int = 50    # 중복 해소 최대 추가 변이 횟수\n    seed: int = 42\n    weights: Tuple[float, ...] = WEIGHT_PRESETS[\"base\"]\n    g0: Tuple[float, ...] = G0_DEFAULT\n    g_star: Tuple[float, ...] = G_STAR_DEFAULT\n\n\n@dataclass\nclass GAResult:\n    best: Chromosome\n    history: List[Tuple[int, float, float, int]] = field(default_factory=list)\n    # history 원소: (generation, best_cost, avg_cost, distinct_schedules)\n    evaluations: int = 0\n    elapsed_sec: float = 0.0\n    stopped_gen: int = 0\n\n\ndef run_ga(cfg: GAConfig) -> GAResult:\n    \"\"\"유전 알고리즘 실행\"\"\"\n    t0 = time.perf_counter()\n    rng = random.Random(cfg.seed)\n    evals = 0\n\n    pop = init_population(cfg.pop_size, rng, dedup=cfg.dedup)\n    for ind in pop:\n        evaluate_chrom(ind, cfg.weights, cfg.g0, cfg.g_star)\n        evals += 1\n\n    pop.sort(key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n    best = pop[0].clone()\n    last_improve_gen = 0\n\n    distinct_0 = len({schedule_key(ind) for ind in pop})\n    avg_0 = sum(c.cost for c in pop if c.cost is not None) / len(pop)\n    history: List[Tuple[int, float, float, int]] = [\n        (0, float(best.cost), avg_0, distinct_0)\n    ]\n\n    stopped_gen = 0\n    for gen in range(1, cfg.max_gens + 1):\n        pop.sort(key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n\n        next_pop: List[Chromosome] = []\n        seen_keys = set()\n\n        # 엘리트 보존 (상위 e개)\n        for i in range(cfg.elite_size):\n            elite_copy = pop[i].clone()\n            next_pop.append(elite_copy)\n            seen_keys.add(schedule_key(elite_copy))\n\n        # 나머지 N - e개 자식 생성\n        while len(next_pop) < cfg.pop_size:\n            p1 = tournament_select(pop, cfg.tournament_k, rng)\n            p2 = tournament_select(pop, cfg.tournament_k, rng)\n            child = crossover(p1, p2, cfg.pc, rng)\n            mutate(child, cfg.pm, rng)\n\n            if cfg.dedup:\n                attempts = 0\n                while schedule_key(child) in seen_keys and attempts < cfg.max_dedup_muts:\n                    mutate_once(child, rng)\n                    attempts += 1\n\n            seen_keys.add(schedule_key(child))\n            evaluate_chrom(child, cfg.weights, cfg.g0, cfg.g_star)\n            evals += 1\n            next_pop.append(child)\n\n        pop = next_pop\n        cur_best = min(pop, key=lambda c: c.cost if c.cost is not None else float(\"inf\"))\n        assert cur_best.cost is not None and best.cost is not None\n        if cur_best.cost < best.cost - 1e-12:\n            best = cur_best.clone()\n            last_improve_gen = gen\n\n        distinct_cnt = len({schedule_key(ind) for ind in pop})\n        avg_cost = sum(c.cost for c in pop if c.cost is not None) / len(pop)\n        history.append((gen, float(best.cost), avg_cost, distinct_cnt))\n        stopped_gen = gen\n\n        if cfg.stall_gens > 0 and (gen - last_improve_gen) >= cfg.stall_gens:\n            break\n\n    assert is_feasible(decode(best))\n    elapsed = time.perf_counter() - t0\n    return GAResult(\n        best=best,\n        history=history,\n        evaluations=evals,\n        elapsed_sec=elapsed,\n        stopped_gen=stopped_gen,\n    )\n\n\n# ---------------------------------------------------------------------------\n# 6. 기준값 측정, 교수님 예시 대조, 리포트 포맷팅\n# ---------------------------------------------------------------------------\ndef measure_random_baseline(n_samples: int = 1000, seed: int = 0) -> Tuple[float, ...]:\n    \"\"\"무작위 (sigma, pi, h) 일정 n_samples개의 G1~G7 평균 측정\"\"\"\n    rng = random.Random(seed)\n    sums = [0.0] * 7\n    for _ in range(n_samples):\n        g_vals = compute_g_values(decode(random_chromosome(rng)))\n        for k in range(7):\n            sums[k] += g_vals[k]\n    return tuple(s / n_samples for s in sums)\n\n\ndef professor_example_schedule() -> Schedule:\n    \"\"\"예시 보고서 Table 4의 10일 경기 일정 (원정 @ 홈 -> (홈, 원정))\"\"\"\n    idx = {name: i for i, name in enumerate(TEAMS)}\n    raw = [\n        [(\"삼미\", \"MBC\"), (\"해태\", \"OB\"), (\"롯데\", \"삼성\")],  # 1일차\n        [(\"OB\", \"삼성\"), (\"해태\", \"삼미\"), (\"롯데\", \"MBC\")],  # 2일차\n        [(\"OB\", \"삼미\"), (\"삼성\", \"MBC\"), (\"롯데\", \"해태\")],  # 3일차\n        [(\"MBC\", \"삼미\"), (\"OB\", \"해태\"), (\"삼성\", \"롯데\")],  # 4일차\n        [(\"MBC\", \"삼성\"), (\"삼미\", \"OB\"), (\"해태\", \"롯데\")],  # 5일차\n        [(\"삼미\", \"삼성\"), (\"OB\", \"롯데\"), (\"해태\", \"MBC\")],  # 6일차\n        [(\"OB\", \"MBC\"), (\"삼성\", \"해태\"), (\"롯데\", \"삼미\")],  # 7일차\n        [(\"MBC\", \"해태\"), (\"삼성\", \"삼미\"), (\"롯데\", \"OB\")],  # 8일차\n        [(\"MBC\", \"롯데\"), (\"삼미\", \"해태\"), (\"삼성\", \"OB\")],  # 9일차\n        [(\"MBC\", \"OB\"), (\"삼미\", \"롯데\"), (\"해태\", \"삼성\")],  # 10일차\n    ]\n    return tuple(tuple((idx[h], idx[a]) for h, a in day) for day in raw)\n\n\ndef _disp_width(text: str) -> int:\n    \"\"\"한글(전각) 2칸, 영문/숫자(반각) 1칸 기준 출력 폭 계산\"\"\"\n    return sum(2 if unicodedata.east_asian_width(ch) in (\"W\", \"F\") else 1 for ch in text)\n\n\ndef _pad_left(text: str, width: int) -> str:\n    return text + \" \" * max(0, width - _disp_width(text))\n\n\ndef _pad_right(text: str, width: int) -> str:\n    return \" \" * max(0, width - _disp_width(text)) + text\n\n\ndef _pad_center(text: str, width: int) -> str:\n    gap = max(0, width - _disp_width(text))\n    left = gap // 2\n    return \" \" * left + text + \" \" * (gap - left)\n\n\ndef format_report(res: GAResult) -> str:\n    \"\"\"최종 10일 일정, 구단별 공정성 지표, G1~G7, 제약 검증 요약 포맷팅\"\"\"\n    best = res.best\n    sched = decode(best)\n    assert best.g_vals is not None and best.cost is not None and best.fitness is not None\n    g_vals = best.g_vals\n    stats = detailed_team_stats(sched)\n    d_t = stats[\"D_t\"]\n    rho_t = stats[\"rho_t\"]\n    w_t = stats[\"W_t\"]\n    l_t = stats[\"L_t\"]\n    breaks = stats[\"breaks\"]\n    max_streak = stats[\"max_streak\"]\n    patterns = stats[\"patterns\"]\n\n    prof_cost, _, _ = evaluate_schedule(professor_example_schedule())\n\n    lines: List[str] = []\n    lines.append(\"=\" * 82)\n    lines.append(\n        f\"최적 일정 탐색 결과 (종료 세대: {res.stopped_gen}, 총 평가: {res.evaluations:,}회, {res.elapsed_sec:.2f}s)\"\n    )\n    lines.append(\n        f\"정규화 비용 C(x) = {best.cost:.4f} (교수님 예시 {prof_cost:.4f} 대비 {best.cost - prof_cost:+.4f}) \"\n        f\"| 보조 적합도 f(x) = {best.fitness:.4f}\"\n    )\n    lines.append(f\"독립 제약 검증 is_feasible(): {is_feasible(sched)}\")\n    lines.append(\"-\" * 82)\n    lines.append(\n        f\"G1(총이동)={g_vals[0]:.0f}km | G2(최대)={g_vals[1]:.0f}km | \"\n        f\"G3(비율std)={g_vals[2]:.4f} | G4(최장원정)={g_vals[3]:.0f}km | \"\n        f\"G5(>=300km)={g_vals[4]:.0f}회 | G6(3연속초과)={g_vals[5]:.0f} | G7(리피터)={g_vals[6]:.0f}\"\n    )\n    lines.append(\"-\" * 82)\n    lines.append(\"[10일 경기 스케줄] (표기: 원정 @ 홈)\")\n    for d, day in enumerate(sched, 1):\n        day_str = \"   \".join(_pad_left(f\"{TEAMS[a]} @ {TEAMS[h]}\", 12) for h, a in day)\n        lines.append(f\"  {d:2d}일차: {day_str.rstrip()}\")\n    lines.append(\"-\" * 82)\n    lines.append(\"[구단별 공정성 지표]\")\n\n    headers = [\"구단\", \"홈/원정 패턴\", \"총이동(km)\", \"D/B비율\", \"최장원정\", \">=300km\", \"브레이크\", \"최장연속\"]\n    col_widths = [4, 12, 10, 7, 8, 7, 8, 8]\n    header_line = \"  \" + \" | \".join(_pad_center(h, w) for h, w in zip(headers, col_widths))\n    lines.append(header_line)\n    lines.append(\"  \" + \"-+-\".join(\"-\" * w for w in col_widths))\n\n    for t in range(N_TEAMS):\n        cells = [\n            _pad_left(TEAMS[t], col_widths[0]),\n            _pad_center(patterns[t], col_widths[1]),\n            _pad_right(f\"{d_t[t]:,d}\", col_widths[2]),\n            _pad_right(f\"{rho_t[t]:.3f}\", col_widths[3]),\n            _pad_right(f\"{w_t[t]:,d}\", col_widths[4]),\n            _pad_right(f\"{l_t[t]}\", col_widths[5]),\n            _pad_right(f\"{breaks[t]}\", col_widths[6]),\n            _pad_right(f\"{max_streak[t]}\", col_widths[7]),\n        ]\n        lines.append(\"  \" + \" | \".join(cells))\n\n    lines.append(\"=\" * 82)\n    return \"\\n\".join(lines)\n\n\n# ---------------------------------------------------------------------------\n# 7. 구현 완료 판정 테스트 8종 (OR2_GA_설계.md 기준)\n# ---------------------------------------------------------------------------\ndef run_completion_tests() -> None:\n    \"\"\"설계 문서의 8가지 완료 판정 테스트 실행\"\"\"\n    rng = random.Random(12345)\n\n    # Test 1: 염색체 및 카드 무결성 (초기화, OX/1점 교차, 4종 변이)\n    for _ in range(100):\n        p1 = random_chromosome(rng)\n        p2 = random_chromosome(rng)\n        c = crossover(p1, p2, 1.0, rng)\n        mutate(c, 1.0, rng)\n        assert sorted(c.sigma) == [0, 1, 2, 3, 4, 5]\n        assert sorted(c.pi) == list(range(10))\n        assert len(c.h) == 15 and all(b in (0, 1) for b in c.h)\n        assert is_feasible(decode(c))\n\n    # Test 2 & 3: 무작위 염색체 1,000개 실행 가능성 및 표현형 다양성 비율\n    decoded_keys = set()\n    for _ in range(1000):\n        ind = random_chromosome(rng)\n        sched = decode(ind)\n        assert is_feasible(sched)\n        decoded_keys.add(schedule_key(ind))\n    unique_ratio = len(decoded_keys) / 1000.0\n    assert unique_ratio > 0.99, f\"다양성 비율이 지나치게 낮음: {unique_ratio}\"\n\n    # Test 4: 비용 계산 재현성 및 교수님 예시 일정 대조\n    prof_sched = professor_example_schedule()\n    assert is_feasible(prof_sched)\n    c1, f1, g1 = evaluate_schedule(prof_sched)\n    c2, f2, _ = evaluate_schedule(prof_sched)\n    assert math.isclose(c1, c2, rel_tol=0.0, abs_tol=1e-12)\n    assert math.isclose(f1, f2, rel_tol=0.0, abs_tol=1e-12)\n    assert g1[0] == 9280.0 and g1[1] == 1585.0\n    assert abs(c1 - 2.013) < 0.01, f\"교수님 예시 v2 비용 불일치: {c1}\"\n\n    # Test 5 & 6 & 7: 세대 크기(60), 엘리트 단조성, 동일 시드 재현성\n    res1 = run_ga(GAConfig(pop_size=60, max_gens=50, stall_gens=0, seed=42))\n    res2 = run_ga(GAConfig(pop_size=60, max_gens=50, stall_gens=0, seed=42))\n    assert len(res1.history) == 51\n    for gen in range(1, len(res1.history)):\n        assert res1.history[gen][1] <= res1.history[gen - 1][1] + 1e-12\n    assert res1.best.cost == res2.best.cost\n    assert decode(res1.best) == decode(res2.best)\n\n    # Test 8: 최종 출력 생성 검증\n    rep = format_report(res1)\n    assert \"10일 경기 스케줄\" in rep and \"True\" in rep\n\n    print(\n        f\"[완료 판정 테스트 8/8 통과] \"\n        f\"무작위 1,000개 실행가능 100% (고유 일정 {len(decoded_keys)}/1000={unique_ratio:.3f}), \"\n        f\"교수님 예시 C(x)={c1:.4f}\"\n    )\n\n\n# ---------------------------------------------------------------------------\n# 8. 실험 실행기 (다중 시드 / 교차율 비교 / 가중치 민감도)\n# ---------------------------------------------------------------------------\ndef run_multi_seed(\n    seeds: int = 8,\n    pc: float = 0.9,\n    pm: float = 0.3,\n    pop_size: int = 60,\n    max_gens: int = 500,\n    stall_gens: int = 150,\n    dedup: bool = True,\n    weights: Tuple[float, ...] = WEIGHT_PRESETS[\"base\"],\n    label: str = \"기본 설정\",\n) -> GAResult:\n    \"\"\"다중 시드 실행 후 통계 및 최선 해 반환\"\"\"\n    costs: List[float] = []\n    distincts: List[int] = []\n    best_res: Optional[GAResult] = None\n    t0 = time.perf_counter()\n\n    for s in range(seeds):\n        cfg = GAConfig(\n            pop_size=pop_size,\n            max_gens=max_gens,\n            stall_gens=stall_gens,\n            pc=pc,\n            pm=pm,\n            tournament_k=3,\n            elite_size=2,\n            dedup=dedup,\n            seed=s,\n            weights=weights,\n        )\n        res = run_ga(cfg)\n        assert res.best.cost is not None\n        costs.append(res.best.cost)\n        distincts.append(res.history[-1][3])\n        if best_res is None or res.best.cost < best_res.best.cost:  # type: ignore[operator]\n            best_res = res\n\n    assert best_res is not None\n    mean_c = statistics.mean(costs)\n    std_c = statistics.stdev(costs) if len(costs) > 1 else 0.0\n    print(\n        f\"[{label} | seeds={seeds}, pc={pc}, pm={pm}, N={pop_size}, G={max_gens}, stall={stall_gens}] \"\n        f\"평균={mean_c:.3f} ± {std_c:.3f} | 최선={min(costs):.3f} | 최악={max(costs):.3f} | \"\n        f\"고유일정={statistics.mean(distincts):.1f}/{pop_size} | 시간={time.perf_counter()-t0:.1f}s\"\n    )\n    return best_res\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"O.R.2 프로야구 10일 일정 GA (통합 최적안)\")\n    parser.add_argument(\"--mode\", choices=[\"demo\", \"single\", \"ablation\"], default=\"demo\")\n    parser.add_argument(\"--seeds\", type=int, default=8)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--pc\", type=float, default=0.9)\n    parser.add_argument(\"--pm\", type=float, default=0.3)\n    parser.add_argument(\"--pop-size\", type=int, default=60)\n    parser.add_argument(\"--max-gens\", type=int, default=500)\n    parser.add_argument(\"--stall\", type=int, default=150)\n    parser.add_argument(\"--weight\", choices=list(WEIGHT_PRESETS.keys()), default=\"base\")\n    args, _ = parser.parse_known_args()\n\n    run_completion_tests()\n\n    if args.mode == \"single\":\n        res = run_ga(\n            GAConfig(\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=args.stall,\n                pc=args.pc,\n                pm=args.pm,\n                seed=args.seed,\n                weights=WEIGHT_PRESETS[args.weight],\n            )\n        )\n        print(format_report(res))\n    elif args.mode == \"ablation\":\n        print(\"\\n--- 교차 확률 비교 (4장 보고용: pc = 0.0, 0.5, 0.9) ---\")\n        for pc in (0.0, 0.5, 0.9):\n            run_multi_seed(\n                seeds=args.seeds,\n                pc=pc,\n                pm=args.pm,\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=0,\n                label=f\"pc={pc}\",\n            )\n        print(\"\\n--- 가중치 민감도 비교 (base / fairness / efficiency) ---\")\n        for name, w in WEIGHT_PRESETS.items():\n            best_w = run_multi_seed(\n                seeds=args.seeds,\n                pc=args.pc,\n                pm=args.pm,\n                pop_size=args.pop_size,\n                max_gens=args.max_gens,\n                stall_gens=args.stall,\n                weights=w,\n                label=f\"weight={name}\",\n            )\n            assert best_w.best.g_vals is not None\n            print(f\"  -> {name} 최선 G1~G7: {[round(x, 4) for x in best_w.best.g_vals]}\")\n    else:\n        print(\"\\n--- 통합 최적안 다중 시드 탐색 ---\")\n        best_res = run_multi_seed(\n            seeds=args.seeds,\n            pc=args.pc,\n            pm=args.pm,\n            pop_size=args.pop_size,\n            max_gens=args.max_gens,\n            stall_gens=args.stall,\n            weights=WEIGHT_PRESETS[args.weight],\n            label=\"채택안(유진+규민+진우 통합)\",\n        )\n        print(format_report(best_res))\n\n\nif __name__ == \"__main__\":\n    main()\n"

def sync_project_files():
    cwd = Path.cwd()
    # 이미 저장소 루트 안에 있는 경우
    if (cwd / "gyumin_improved.py").exists() and (cwd / "gyumin_original.py").exists():
        print(f"✅ 로컬 저장소 파일 확인 완료: {cwd}")
        return cwd

    # 하위 폴더로 이미 클론되어 있는 경우
    sub = cwd / REPO_NAME
    if (sub / "gyumin_improved.py").exists() and (sub / "gyumin_original.py").exists():
        os.chdir(sub)
        print(f"✅ 기존 클론 폴더로 이동 완료: {Path.cwd()}")
        return Path.cwd()

    # GitHub에서 clone 시도
    print(f"🔄 GitHub 저장소 클론 시도: {REPO_URL}")
    proc = subprocess.run(["git", "clone", "--depth", "1", REPO_URL], capture_output=True, text=True)
    if proc.returncode == 0 and (sub / "gyumin_improved.py").exists():
        os.chdir(sub)
        print(f"✅ GitHub 클론 성공! 작업 디렉터리: {Path.cwd()}")
        return Path.cwd()

    # 오프라인 / 미푸시 상태일 때는 노트북 내장 소스로 자동 복원
    print("⚠️ GitHub 클론을 건너뛰고 노트북 내장 코드로 로컬 파일을 생성합니다.")
    (cwd / "gyumin_improved.py").write_text(_FALLBACK_IMPROVED, encoding="utf-8")
    (cwd / "gyumin_original.py").write_text(_FALLBACK_ORIGINAL, encoding="utf-8")
    print(f"✅ 내장 소스 파일 생성 완료: {cwd}")
    return cwd

WORK_DIR = sync_project_files()
print("📁 파일 목록:", sorted(p.name for p in WORK_DIR.iterdir() if p.suffix in (".py", ".csv", ".xlsx", ".md", ".ipynb")))

In [ ]:
#@title 🧪 2. GA 모듈 로드 및 8종 완료 판정 테스트 실행 { display-mode: "form" }
import importlib
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

if not (Path("gyumin_improved.py").exists() and Path("gyumin_original.py").exists()):
    if "sync_project_files" in globals():
        sync_project_files()
    else:
        raise FileNotFoundError("먼저 1번 셀(GitHub 저장소 연동 및 파일 동기화)을 실행해 주세요.")

import gyumin_improved
import gyumin_original

importlib.reload(gyumin_improved)
importlib.reload(gyumin_original)

print("▶ [규민 개량본 gyumin_improved.py] 완료 판정 테스트:")
gyumin_improved.run_completion_tests()

print("▶ [규민 원본 gyumin_original.py] 완료 판정 테스트:")
gyumin_original.run_completion_tests()

In [ ]:
#@title ⚙️ 3. GA 탐색 실행 (파라미터 조정 가능) { display-mode: "form" }
VERSION = "gyumin_improved (pc=0.6)" #@param ["gyumin_improved (pc=0.6)", "gyumin_original (pc=0.9)"]
MODE = "best_repro (최선 해 재현)" #@param ["best_repro (최선 해 재현)", "single (단일 시드)", "multi_seed (다중 시드)", "ablation (교차율·가중치 비교)"]
SEED = 8 #@param {type:"integer"}
SEEDS = 8 #@param {type:"integer"}
POP_SIZE = 60 #@param {type:"integer"}
MAX_GENS = 500 #@param {type:"integer"}
STALL_GENS = 150 #@param {type:"integer"}
WEIGHT_PRESET = "base" #@param ["base", "fairness", "efficiency"]

is_improved = VERSION.startswith("gyumin_improved")
active_mod = gyumin_improved if is_improved else gyumin_original
default_pc = 0.6 if is_improved else 0.9
weights = active_mod.WEIGHT_PRESETS[WEIGHT_PRESET]

if MODE.startswith("best_repro"):
    repro_seed = 8 if is_improved else 3
    print(f"🎯 [{VERSION}] 최선 일정 재현 실행 (seed={repro_seed}, pc={default_pc})")
    best_res = active_mod.run_ga(
        active_mod.GAConfig(
            pop_size=60,
            max_gens=500,
            stall_gens=150,
            pc=default_pc,
            pm=0.3,
            seed=repro_seed,
            weights=weights,
        )
    )
    print(active_mod.format_report(best_res))

elif MODE.startswith("single"):
    print(f"🔍 [{VERSION}] 단일 시드 탐색 (seed={SEED}, pc={default_pc}, weight={WEIGHT_PRESET})")
    best_res = active_mod.run_ga(
        active_mod.GAConfig(
            pop_size=POP_SIZE,
            max_gens=MAX_GENS,
            stall_gens=STALL_GENS,
            pc=default_pc,
            pm=0.3,
            seed=SEED,
            weights=weights,
        )
    )
    print(active_mod.format_report(best_res))

elif MODE.startswith("multi_seed"):
    print(f"📊 [{VERSION}] 다중 시드 탐색 (seeds=0..{SEEDS - 1}, pc={default_pc}, weight={WEIGHT_PRESET})")
    best_res = active_mod.run_multi_seed(
        seeds=SEEDS,
        pc=default_pc,
        pm=0.3,
        pop_size=POP_SIZE,
        max_gens=MAX_GENS,
        stall_gens=STALL_GENS,
        weights=weights,
        label=VERSION,
    )
    print(active_mod.format_report(best_res))

else:
    print(f"🔬 [{VERSION}] 교차 확률 및 가중치 민감도 실험 (seeds=0..{SEEDS - 1})")
    print("\n--- 교차 확률 비교 (pc = 0.0, 0.5, 0.9) ---")
    for pc_val in (0.0, 0.5, 0.9):
        active_mod.run_multi_seed(
            seeds=SEEDS,
            pc=pc_val,
            pm=0.3,
            pop_size=POP_SIZE,
            max_gens=MAX_GENS,
            stall_gens=0,
            label=f"pc={pc_val}",
        )
    print("\n--- 가중치 민감도 비교 (base / fairness / efficiency, 동일 base 가중치 재평가 포함) ---")
    base_weights = active_mod.WEIGHT_PRESETS["base"]
    best_res = None
    best_base_cost = float("inf")
    best_w_name = ""
    for w_name, w_vec in active_mod.WEIGHT_PRESETS.items():
        res_w = active_mod.run_multi_seed(
            seeds=SEEDS,
            pc=default_pc,
            pm=0.3,
            pop_size=POP_SIZE,
            max_gens=MAX_GENS,
            stall_gens=STALL_GENS,
            weights=w_vec,
            label=f"weight={w_name}",
        )
        base_cost, _, _ = active_mod.evaluate_schedule(
            active_mod.decode(res_w.best), weights=base_weights
        )
        if base_cost < best_base_cost:
            best_base_cost = base_cost
            best_res = res_w
            best_w_name = w_name
        print(
            f"  -> {w_name} 최선: 자체 비용 C_{w_name}={res_w.best.cost:.4f} | "
            f"base 기준 재평가 C_base={base_cost:.4f} | "
            f"G1~G7={[round(x, 4) for x in res_w.best.g_vals]}"
        )
    print(f"\n🏆 [동일 기준(base 가중치) 재평가 최종안 선정: weight={best_w_name}, C_base={best_base_cost:.4f}]")
    if best_w_name != "base":
        best_res_base = active_mod.GAResult(
            best=best_res.best.clone(),
            history=best_res.history,
            evaluations=best_res.evaluations,
            elapsed_sec=best_res.elapsed_sec,
            stopped_gen=best_res.stopped_gen,
        )
        c_b, f_b, g_b = active_mod.evaluate_schedule(
            active_mod.decode(best_res_base.best), weights=base_weights
        )
        best_res_base.best.cost = c_b
        best_res_base.best.fitness = f_b
        best_res_base.best.g_vals = g_b
        best_res = best_res_base
    print(active_mod.format_report(best_res))

In [ ]:
#@title 📋 4. 교수님 템플릿 양식(구장별 대진표 & 일별 이동거리표) 출력 및 CSV 저장 { display-mode: "form" }
import csv
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

if "best_res" not in globals() or best_res is None:
    raise RuntimeError("먼저 3번 셀(GA 탐색 실행)을 실행해 주세요.")

CITIES = ("서울", "인천", "대전", "광주", "대구", "부산")
TEAMS = active_mod.TEAMS
DIST = active_mod.DIST
N_TEAMS = active_mod.N_TEAMS
N_DAYS = active_mod.N_DAYS

sched = active_mod.decode(best_res.best)

# 1) 구장별 대진표 (Schedule)
sched_rows = []
for d_idx, day in enumerate(sched, start=1):
    row = {"Day": f"Day {d_idx}"}
    for city in CITIES:
        row[city] = ""
    for home, away in day:
        row[CITIES[home]] = f"{TEAMS[away]} @ {TEAMS[home]}"
    sched_rows.append(row)

# 2) 구단별 일별 이동거리표 (Distance)
loc = [[t] * (N_DAYS + 2) for t in range(N_TEAMS)]
for d_idx, day in enumerate(sched):
    for home, away in day:
        loc[home][d_idx + 1] = home
        loc[away][d_idx + 1] = home

dist_rows = []
for r in range(N_DAYS + 1):
    label = f"Day {r + 1}" if r < N_DAYS else "복귀"
    row = {"Day": label}
    for t in range(N_TEAMS):
        row[TEAMS[t]] = DIST[loc[t][r]][loc[t][r + 1]]
    dist_rows.append(row)

if pd is not None:
    df_sched = pd.DataFrame(sched_rows).set_index("Day")
    df_dist = pd.DataFrame(dist_rows).set_index("Day")
    print(f"🏟️ [10일 경기 일정 - 구장별 표기] (C = {best_res.best.cost:.4f})")
    display(df_sched)
    print("\n🚌 [구단별 일일 및 복귀 이동거리 (km)]")
    display(df_dist)
else:
    print("Schedule:", sched_rows)
    print("Distance:", dist_rows)

# 교수님 템플릿 CSV 형식으로 저장
out_csv = Path("schedule_result.csv")
with out_csv.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["Schedule", "", "", "", "", "", ""])
    writer.writerow(["", *CITIES])
    for r in sched_rows:
        writer.writerow([r["Day"], *(r[c] for c in CITIES)])
    for _ in range(5):
        writer.writerow(["", "", "", "", "", "", ""])
    writer.writerow(["Distance", "", "", "", "", "", ""])
    writer.writerow(["", "", "", "", "", "", ""])
    writer.writerow(["", *TEAMS])
    for r in dist_rows:
        writer.writerow([r["Day"], *(r[t] for t in TEAMS)])
    writer.writerow(["", "", "", "", "", "", ""])
    writer.writerow([f"C = {best_res.best.cost:.4f} (v2), {VERSION}", "", "", "", "", "", ""])

print(f"💾 CSV 저장 완료: {out_csv.resolve()}")

In [ ]:
#@title 📈 5. 세대별 수렴 곡선 및 개체군 다양성 시각화 { display-mode: "form" }
import matplotlib.pyplot as plt
from matplotlib import font_manager

if "best_res" not in globals() or best_res is None:
    raise RuntimeError("먼저 3번 셀(GA 탐색 실행)을 실행해 주세요.")

# 한글 폰트 탐색 (미설치 환경에서는 깨짐 방지를 위해 영문 라벨 사용)
_KOR_FONT_CANDIDATES = ("NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR")
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
_selected_kor_font = next((name for name in _KOR_FONT_CANDIDATES if name in _available_fonts), None)
if _selected_kor_font:
    plt.rcParams["font.family"] = _selected_kor_font
plt.rcParams["axes.unicode_minus"] = False

gens = [h[0] for h in best_res.history]
best_costs = [h[1] for h in best_res.history]
avg_costs = [h[2] for h in best_res.history]
distincts = [h[3] for h in best_res.history]

prof_cost, _, _ = active_mod.evaluate_schedule(active_mod.professor_example_schedule())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)

# Subplot 1: Cost Convergence
ax1 = axes[0]
ax1.plot(gens, best_costs, label=f"Best C(x) = {best_costs[-1]:.4f}", color="#1f77b4", linewidth=2.2)
ax1.plot(gens, avg_costs, label="Population Avg C(x)", color="#ff7f0e", alpha=0.75, linewidth=1.4)
ax1.axhline(prof_cost, color="#d62728", linestyle="--", linewidth=1.3, label=f"Professor Example ({prof_cost:.3f})")
ax1.set_xlabel("Generation" if not _selected_kor_font else "세대 (Generation)")
ax1.set_ylabel("Normalized Cost C(x)" if not _selected_kor_font else "정규화 비용 C(x)")
ax1.set_title(
    f"Convergence Curve ({VERSION})" if not _selected_kor_font else f"GA 수렴 곡선 ({VERSION})"
)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Subplot 2: Phenotype Diversity
ax2 = axes[1]
ax2.plot(gens, distincts, color="#2ca02c", linewidth=1.8, label="Distinct Schedules")
ax2.set_ylim(0, max(distincts) * 1.12 if distincts else 65)
ax2.set_xlabel("Generation" if not _selected_kor_font else "세대 (Generation)")
ax2.set_ylabel("Distinct Schedules" if not _selected_kor_font else "고유 일정 수")
ax2.set_title(
    "Population Phenotype Diversity" if not _selected_kor_font else "개체군 표현형 다양성 (고유 일정 수)"
)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()